In [ ]:
#!/usr/bin/env python
# coding: utf-8

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv("datasets/reduced_lending_club.csv")
df.shape

In [ ]:
df.info()

In [ ]:
percent_missing = df.isnull().sum() * 100 / len(df)
missing_value_df = pd.DataFrame({'column_name': df.columns,
                                 'percent_missing': percent_missing})
missing_value_df.sort_values('percent_missing', inplace=True, ascending=False)
missing_value_df

In [ ]:
print(df.sample())

In [ ]:
df.columns.tolist()

**Loan Details:**<br>
loan_amnt,<br>
funded_amnt,<br>
term,<br>
int_rate,<br>
installment,<br>
purpose,<br>
application_type<br>
,grade<br>
,sub_grade<br>
<br>
**Borrower Profile:**<br>
annual_inc,<br>
emp_length,<br>
home_ownership,<br>
verification_status,<br>
addr_state,<br>
dti<br>
<br>
**Credit History:**<br>
earliest_cr_line,<br>
pub_rec,<br>
pub_rec_bankruptcies,<br>
tax_liens<br>
<br>
**Deliquency History:**<br>
acc_now_delinq,<br>
delinq_2yrs,<br>
delinq_amnt,<br>
mths_since_last_delinq,<br>
collections_12_mths_ex_med,<br>
chargeoff_within_12_mths,<br>
num_accts_ever_120_pd,<br>
num_tl_30dpd,<br>
num_tl_90g_dpd_24m<br>
<br>
**Financial Stress:**<br>
revol_bal,<br>
revol_util,<br>
bc_util,<br>
avg_cur_bal,<br>
tot_cur_bal,<br>
total_bal_ex_mort,<br>
total_bc_limit,<br>
percent_bc_gt_75<br>
<br>
**Account Stability:**<br>
open_acc,<br>
total_acc,<br>
mort_acc,<br>
pct_tl_nvr_dlq<br>
<br>
**Account Activity:**<br>
inq_last_6mths,<br>
acc_open_past_24mths<br>
<br>
**Targets:**<br>
loan_status,<br>
recoveries,<br>
collection_recovery_fee

In [ ]:
df['net_recovery'] = (df['recoveries']-df['collection_recovery_fee'])
df['recovery_flag'] = (df['recoveries']>0).astype(int)
df['income_to_loan_ratio'] = (df['annual_inc']/df['loan_amnt'])
df['earliest_cr_line'] = pd.to_datetime(df['earliest_cr_line'],format='%b-%Y',errors='coerce')
df['credit_age_years'] = (pd.Timestamp.today() -pd.to_datetime(df['earliest_cr_line'])).dt.days / 365.25

In [ ]:
print(df[['earliest_cr_line', 'credit_age_years']].head())

In [ ]:
df = df.drop(columns=['earliest_cr_line'])

In [ ]:
df['has_prior_delinq'] = (df["mths_since_last_delinq"].notnull().astype(int))
df["mths_since_last_delinq"] = (df["mths_since_last_delinq"].fillna(999))

In [ ]:
num_cols = df.select_dtypes(include="number").columns.tolist()
cat_cols = df.select_dtypes(include=str).columns.tolist()

In [ ]:
print(num_cols)
print(cat_cols)

In [ ]:
for col in num_cols:
    if df[col].isnull().sum()>0:
        df[col] = df[col].fillna(df[col].median())

In [ ]:
df['emp_length'] = (
    df['emp_length']
    .fillna(df['emp_length'].mode()[0])
)

In [ ]:
df.isnull().sum()

In [ ]:
df['delinq_score'] = (
    df['acc_now_delinq']
    + df['delinq_2yrs']
    + df['num_tl_90g_dpd_24m']
    + df['num_accts_ever_120_pd']
)
df['stress_score'] = (
    df['revol_util']
    + df['bc_util']
)

In [ ]:
drop_features = [
    'recoveries',
    'collection_recovery_fee',
    'net_recovery',
    'recovery_flag',
    'loan_status'
]

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X = df.drop(columns=drop_features)
y = df['recovery_flag']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,stratify=y,random_state=14)

In [ ]:
from catboost import CatBoostClassifier

In [ ]:
cat_cols = X.select_dtypes(include=str).columns.tolist()

In [ ]:
clf = CatBoostClassifier(
    iterations=500,
    depth=6,
    learning_rate=0.05,
    eval_metric='AUC',
    verbose=100,
    auto_class_weights='Balanced'
)

In [ ]:
clf.fit(
    X_train,
    y_train,
    cat_features=cat_cols
)

In [ ]:
from sklearn.metrics import roc_auc_score

In [ ]:
pred_probs = clf.predict_proba(X_test)[:,1]

In [ ]:
auc = roc_auc_score(y_test,pred_probs)

In [ ]:
print("ROC_AUC:", auc)

In [ ]:
from sklearn.metrics import classification_report

In [ ]:
preds = clf.predict(X_test)
print(classification_report(y_test,preds))

In [ ]:
feature_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': clf.feature_importances_
})

In [ ]:
feature_importance = feature_importance.sort_values(
    'importance',
    ascending=False
)

In [ ]:
print(feature_importance.head(20))

In [ ]:
df['application_type'].value_counts()

In [ ]:
from sklearn.calibration import CalibrationDisplay

In [ ]:
CalibrationDisplay.from_predictions(
    y_test,
    pred_probs
)

In [ ]:
# from sklearn.model_selection import train_test_split
# from sklearn.compose import ColumnTransformer
# from sklearn.pipeline import Pipeline
# from sklearn.impute import SimpleImputer
# from sklearn.preprocessing import OneHotEncoder, StandardScaler
# from sklearn.linear_model import LogisticRegression
# from sklearn.metrics import (
# roc_auc_score,
# classification_report
# )

In [ ]:
# categorical_cols = X.select_dtypes(
#     include=['object', 'category']
# ).columns.tolist()

In [ ]:
# numeric_cols = X.select_dtypes(
#     exclude=['object', 'category']
# ).columns.tolist()

In [ ]:
# numeric_transformer = Pipeline(
#     steps=[
#         ('imputer', SimpleImputer(strategy='median')),
#         ('scaler', StandardScaler())
#     ]
# )

In [ ]:
# categorical_transformer = Pipeline(
#     steps=[
#         ('imputer', SimpleImputer(strategy='most_frequent')),
#         (
#             'onehot',
#             OneHotEncoder(
#                 handle_unknown='ignore'
#             )
#         )
#     ]
# )

In [ ]:
# preprocessor = ColumnTransformer(
#     transformers=[
#         ('num', numeric_transformer, numeric_cols),<br>
#         ('cat', categorical_transformer, categorical_cols)<br>
#     ]
# )

In [ ]:
# logreg_pipeline = Pipeline(
#     steps=[
#         ('preprocessor', preprocessor),
#         (
#             'classifier',
#             LogisticRegression(
#                 max_iter=2000,
#                 class_weight='balanced',
#                 random_state=42
#             )
#         )
#     ]
# )

In[26]:

In [ ]:
# logreg_pipeline.fit(
#     X_train,
#     y_train
# )

In[27]:

In [ ]:
# pred_probs = logreg_pipeline.predict_proba(
#     X_test
# )[:, 1]

In [ ]:
# pred_class = logreg_pipeline.predict(
#     X_test
# )

In[28]:

In [ ]:
# print(
#     "ROC-AUC:",
#     roc_auc_score(
#         y_test,
#         pred_probs
#     )
# )

In [ ]:
# print(
#     classification_report(
#         y_test,
#         pred_class
#     )
# )

In [ ]:
# from sklearn.ensemble import RandomForestClassifier

In [ ]:
# rf_pipeline = Pipeline(
#     steps=[
#         ('preprocessor', preprocessor),
#         (
#             'classifier',
#             RandomForestClassifier(
#                 n_estimators=300,
#                 max_depth=10,
#                 random_state=42,
#                 n_jobs=-1
#             )
#         )
#     ]
# )

In [ ]:
# rf_pipeline.fit(X_train, y_train)

In [ ]:
# rf_probs = rf_pipeline.predict_proba(X_test)[:, 1]
# rf_preds = rf_pipeline.predict(X_test)

In [ ]:
# print(
#     "Random Forest ROC-AUC:",
#     roc_auc_score(y_test, rf_probs)
# )

In [ ]:
# print(
#     classification_report(
#         y_test,
#         rf_preds
#     )
# )

In [ ]:
# from xgboost import XGBClassifier<br>
# from sklearn.pipeline import Pipeline<br>
# from sklearn.metrics import roc_auc_score, classification_report

In [ ]:
# xgb_pipeline = Pipeline(<br>
#     steps=[<br>
#         ('preprocessor', preprocessor),
#         (
#             'classifier',
#             XGBClassifier(
#                 n_estimators=500,
#                 max_depth=6,
#                 learning_rate=0.05,
#                 subsample=0.8,
#                 colsample_bytree=0.8,
#                 eval_metric='logloss',
#                 random_state=42,
#                 n_jobs=-1
#             )
#         )
#     ]
# )

In [ ]:
# xgb_pipeline.fit(
#     X_train,
#     y_train
# )

In [ ]:
# xgb_probs = xgb_pipeline.predict_proba(
#     X_test
# )[:, 1]

In [ ]:
# xgb_preds = xgb_pipeline.predict(
#     X_test
# )

In [ ]:
# print(
#     "XGBoost ROC-AUC:",
#     roc_auc_score(
#         y_test,
#         xgb_probs
#     )
# )

In [ ]:
# print(
#     classification_report(
#         y_test,
#         xgb_preds
#     )
# )

In[31]:

In [ ]:
from catboost import Pool, cv

In [ ]:
cat_cols = X.select_dtypes(
    include=['object', 'category']
).columns.tolist()

In [ ]:
train_pool = Pool(
    X,
    y,
    cat_features=cat_cols
)

In [ ]:
params = {
    'loss_function': 'Logloss',
    'eval_metric': 'AUC',
    'iterations': 500,
    'depth': 6,
    'learning_rate': 0.05,
    'auto_class_weights': 'Balanced',
    'random_seed': 14,
    'verbose': False
}

In [ ]:
cv_results = cv(
    train_pool,
    params,
    fold_count=5,
    shuffle=True,
    partition_random_seed=14
)

In [ ]:
print(
    cv_results['test-AUC-mean'].max()
)